## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:
曾静怡

note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
import sys
sys.setrecursionlimit(10000)

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return
    
    idx = 0
    n = data[idx]; idx += 1
    a = data[idx]; idx += 1
    b = data[idx]; idx += 1
    o = data[idx:idx + n]; idx += n
    
    posi = [0] * n
    for i, val in enumerate(o):
        posi[val] = i
    
    ans = []
    
    # 使用局部变量加速
    o_arr = o
    posi_arr = posi
    n_val = n
    a_val = a
    b_val = b
    
    def refresh():
        for i in range(n_val):
            posi_arr[o_arr[i]] = i
    
    def doMagic():
        ans.append(0)
        for i in range(n_val):
            if o_arr[i] == a_val:
                o_arr[i] = b_val
            elif o_arr[i] == b_val:
                o_arr[i] = a_val
        refresh()
    
    def doAdd(x):
        if x == 0:
            return
        ans.append(x)
        for i in range(n_val):
            o_arr[i] = (o_arr[i] + x) % n_val
        refresh()
    
    def doXor(x):
        if x == 0:
            return
        ans.append(-x)
        for i in range(n_val):
            o_arr[i] ^= x
        refresh()
    
    # 计算 l
    diff = (a_val - b_val) % n_val
    if diff == 0:
        l_val = n_val
    else:
        l_val = diff & -diff
    
    # 缓存 calc 结果
    calc_cache = {}
    
    def calc(a_val_calc, b_val_calc):
        key = (a_val_calc, b_val_calc)
        if key in calc_cache:
            return calc_cache[key]
        delta = (b_val_calc - a_val_calc + n_val - l_val + n_val) % n_val
        pa = 0
        pb = 0
        stp = n_val // 2
        while stp >= 2 * l_val:
            if delta >= stp:
                delta -= stp
                pb += stp // 2
            else:
                pa += stp // 2
            stp //= 2
        pa += n_val // 2
        pa += (a_val_calc & (l_val - 1))
        pb += (a_val_calc & (l_val - 1))
        calc_cache[key] = (pa, pb)
        return pa, pb
    
    def doSwap(c, d):
        if (c // l_val) % 2 == (d // l_val) % 2:
            if (c // l_val) % 2 == 0:
                p_val = (c & (l_val - 1)) + l_val
            else:
                p_val = (c & (l_val - 1))
            doSwap(c, p_val)
            doSwap(d, p_val)
            doSwap(c, p_val)
        else:
            pa, pb = calc(a_val, b_val)
            pc, pd = calc(c, d)
            
            doAdd((pc - c + n_val) % n_val)
            doXor(pc ^ pa)
            doAdd((a_val - pa + n_val) % n_val)
            doMagic()
            doAdd((pa - a_val + n_val) % n_val)
            doXor(pc ^ pa)
            doAdd((c - pc + n_val) % n_val)
    
    # Perm 类 - 核心分治逻辑，尽量优化
    class Perm:
        __slots__ = ('a', 'n', 'vec')
        def __init__(self, arr, sz):
            self.a = arr[:]
            self.n = sz
            self.vec = []
        
        def calc(self):
            # 快速检查
            if self.n == 0:
                return True
            seen = [False] * self.n
            for v in self.a:
                if v >= self.n:
                    return False
                seen[v] = True
            for s in seen:
                if not s:
                    return False
            
            if self.n == 1:
                return True
            
            half = self.n // 2
            
            # 分配数组
            b_arr = [0] * half
            c_arr = [0] * half
            a_local = self.a
            for i in range(half):
                b_arr[i] = a_local[i * 2] // 2
                c_arr[i] = a_local[i * 2 + 1] // 2
            
            b_perm = Perm(b_arr, half)
            c_perm = Perm(c_arr, half)
            
            if not b_perm.calc() or not c_perm.calc():
                return False
            
            vec_local = self.vec
            
            if a_local[0] & 1:
                vec_local.append(1 if self.n == 2 else -1)
            
            tb = 0
            for val in b_perm.vec:
                if val > 0:
                    vec_local.append(-1)
                    vec_local.append(1)
                else:
                    vec_local.append(val * 2)
                    tb ^= -val * 2
            
            if tb:
                vec_local.append(-tb)
            
            tc = 0
            for val in c_perm.vec:
                if val > 0:
                    vec_local.append(1)
                    vec_local.append(-1)
                else:
                    vec_local.append(val * 2)
                    tc ^= -val * 2
            
            if (tc & half) != (tb & half):
                for _ in range(self.n // 4):
                    vec_local.append(-1)
                    vec_local.append(1)
            
            if tb >= half:
                tb -= half
            if tc >= half:
                tc -= half
            
            if tb != tc:
                return False
            
            # 合并
            tmp = []
            for val in vec_local:
                if not tmp:
                    tmp.append(val)
                elif val < 0 and tmp[-1] < 0:
                    tmp[-1] = -((-tmp[-1]) ^ (-val))
                    if tmp[-1] == 0:
                        tmp.pop()
                else:
                    tmp.append(val)
            
            self.vec = tmp
            return True
    
    # 第一层
    if l_val > 1:
        perm_arr = [o_arr[i] & (l_val - 1) for i in range(n_val)]
        perm = Perm(perm_arr, l_val)
        if not perm.calc():
            print(-1)
            return
        
        for val in perm.vec:
            if val > 0:
                doAdd(val)
            else:
                doXor(-val)
    
    # 检查同余类
    for i in range(l_val):
        vec_vals = []
        for j in range(i, n_val, l_val):
            vec_vals.append(o_arr[j])
        vec_vals.sort()
        
        c_idx = 0
        ok = True
        for j in range(i, n_val, l_val):
            if vec_vals[c_idx] != j:
                ok = False
                break
            c_idx += 1
        
        if not ok:
            print(-1)
            return
        
        for j in range(i, n_val, l_val):
            if o_arr[j] != j:
                doSwap(j, o_arr[j])
    
    # 验证并输出
    for i in range(n_val):
        if o_arr[i] != i:
            print(-1)
            return
    
    print(len(ans))
    out_lines = []
    for op in ans:
        if op == 0:
            out_lines.append("0")
        elif op > 0:
            out_lines.append(f"2 {op}")
        else:
            out_lines.append(f"1 {-op}")
    sys.stdout.write("\n".join(out_lines))

if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
## add your code here
import sys

def solve():
    # 使用快读处理多组数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    ptr = 0
    while ptr < len(input_data):
        try:
            N = int(input_data[ptr])
            L = int(input_data[ptr+1])
            Maxn = int(input_data[ptr+2])
            S = int(input_data[ptr+3])
            ptr += 4
        except:
            break
            
        stations = []
        for _ in range(N):
            p = int(input_data[ptr])
            c = int(input_data[ptr+1])
            stations.append((p, c))
            ptr += 2
        
        # 0. 基础判定：如果不补给就能跑完
        if Maxn >= L:
            print("Yes")
            continue
            
        # 1. DP 初始化
        # f[s] 代表花费 s 个硬币能到达的最远距离
        f = [-1] * (S + 1)
        f[0] = Maxn
        
        # 2. 补给站按位置排序
        stations.sort()
        
        can_reach = False
        for p, c in stations:
            # 只有当 c <= S 且能在花费范围内转移时才考虑
            # 采用类似 0/1 背包的倒序遍历，防止同一个补给站被重复计费
            for s in range(S - c, -1, -1):
                if f[s] >= p: # 说明能到达这个补给站
                    f[s + c] = max(f[s + c], p + Maxn)
                    if f[s + c] >= L:
                        can_reach = True
                        break
            if can_reach:
                break
        
        if can_reach:
            print("Yes")
        else:
            print("No")

if __name__ == "__main__":
    solve()

## C 最长回文

In [ ]:
## add your code here
import sys


def manacher(s):
    n = len(s)

    # d1[i]：以 i 为中心的奇数回文半径
    d1 = [0] * n
    l, r = 0, -1
    for i in range(n):
        k = 1 if i > r else min(d1[l + r - i], r - i + 1)

        while i - k >= 0 and i + k < n and s[i - k] == s[i + k]:
            k += 1

        d1[i] = k

        if i + k - 1 > r:
            l = i - k + 1
            r = i + k - 1

    # d2[i]：以 i-1 和 i 中间为中心的偶数回文半径
    d2 = [0] * n
    l, r = 0, -1
    for i in range(n):
        k = 0 if i > r else min(d2[l + r - i + 1], r - i + 1)

        while i - k - 1 >= 0 and i + k < n and s[i - k - 1] == s[i + k]:
            k += 1

        d2[i] = k

        if i + k - 1 > r:
            l = i - k
            r = i + k - 1

    return d1, d2


def solve(n, A, B):
    RA = A[::-1]

    MASK = (1 << 64) - 1
    BASE = 911382323

    pw = [1] * (n + 1)
    h1 = [0] * (n + 1)
    h2 = [0] * (n + 1)

    for i in range(n):
        pw[i + 1] = (pw[i] * BASE) & MASK
        h1[i + 1] = (h1[i] * BASE + RA[i]) & MASK
        h2[i + 1] = (h2[i] * BASE + B[i]) & MASK

    def get_hash(h, l, r):
        return (h[r] - (h[l] * pw[r - l] & MASK)) & MASK

    def lcp(p, q):
        """
        求 RA[p:] 和 B[q:] 的最长公共前缀长度
        """
        if p < 0 or q < 0 or p >= n or q >= n:
            return 0

        if RA[p] != B[q]:
            return 0

        hi = min(n - p, n - q)

        if get_hash(h1, p, p + hi) == get_hash(h2, q, q + hi):
            return hi

        lo = 1
        hi -= 1

        while lo < hi:
            mid = (lo + hi + 1) // 2

            if get_hash(h1, p, p + mid) == get_hash(h2, q, q + mid):
                lo = mid
            else:
                hi = mid - 1

        return lo

    d1A, d2A = manacher(A)
    d1B, d2B = manacher(B)

    ans = 1

    # 纯 A / 纯 B 的最长回文子串
    for x in d1A:
        ans = max(ans, 2 * x - 1)
    for x in d2A:
        ans = max(ans, 2 * x)

    for x in d1B:
        ans = max(ans, 2 * x - 1)
    for x in d2B:
        ans = max(ans, 2 * x)

    # 情况一：中间回文段在 B 里面
    # 奇数中心
    for c, r in enumerate(d1B):
        mid_len = 2 * r - 1

        L = c - r + 1
        R = c + r - 1

        # A 的边界是 L，B 右侧从 R + 1 开始继续匹配
        p = n - 1 - L
        q = R + 1

        if p < n and q < n:
            ans = max(ans, mid_len + 2 * lcp(p, q))

    # 偶数中心，r = 0 时对应空中间段
    for c, r in enumerate(d2B):
        mid_len = 2 * r

        L = c - r
        R = c + r - 1

        p = n - 1 - L
        q = R + 1

        if p < n and q < n:
            ans = max(ans, mid_len + 2 * lcp(p, q))

    # 情况二：中间回文段在 A 里面
    # 奇数中心
    for c, r in enumerate(d1A):
        mid_len = 2 * r - 1

        L = c - r + 1
        R = c + r - 1

        # A 左侧从 L - 1 往左匹配，映射到 RA[n - L:]
        # B 从 R 开始匹配
        p = n - L
        q = R

        if p < n and q < n:
            ans = max(ans, mid_len + 2 * lcp(p, q))

    # 偶数中心
    for c, r in enumerate(d2A):
        mid_len = 2 * r

        L = c - r
        R = c + r - 1

        p = n - L
        q = R

        if p < n and 0 <= q < n:
            ans = max(ans, mid_len + 2 * lcp(p, q))

    return ans


def main():
    data = sys.stdin.buffer.read().split()

    n = int(data[0])
    A = data[1]
    B = data[2]

    print(solve(n, A, B))


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
## add your code here
import sys
from array import array


MAXX = 500000 + 5


def bit_add(bit, n, i, v):
    while i <= n:
        bit[i] += v
        i += i & -i


def bit_sum(bit, i):
    s = 0
    while i > 0:
        s += bit[i]
        i -= i & -i
    return s


def bit_kth(bit, n, k):
    """
    找最小 idx，使得 prefix_sum(idx) >= k
    """
    idx = 0
    step = 1 << (n.bit_length() - 1)

    while step:
        nxt = idx + step
        if nxt <= n and bit[nxt] < k:
            idx = nxt
            k -= bit[nxt]
        step >>= 1

    return idx + 1


def main():
    input = sys.stdin.buffer.readline
    ans = []

    while True:
        line = input()
        if not line:
            break

        line = line.strip()
        if not line:
            continue

        m = int(line)

        cnt = bytearray(MAXX)
        last = array('i', [0]) * MAXX

        bit = array('i', [0]) * (m + 2)
        qcnt = 0

        bad = -1

        for i in range(1, m + 1):
            parts = input().split()

            if bad != -1:
                continue

            op = parts[0]

            if op == b'?':
                bit_add(bit, m, i, 1)
                qcnt += 1
                continue

            x = int(parts[1])

            if op == b'I':
                if cnt[x] == 0:
                    cnt[x] = 1
                else:
                    # 需要找一个 last[x] 之后的 ? 当成 O x
                    l = last[x]
                    before = bit_sum(bit, l - 1) if l > 0 else 0

                    if before == qcnt:
                        bad = i
                    else:
                        pos = bit_kth(bit, m, before + 1)
                        bit_add(bit, m, pos, -1)
                        qcnt -= 1
                        # cnt[x] 仍然是 1：
                        # ? 当 O x，当前 I x 又买入

                last[x] = i

            else:
                # op == b'O'
                if cnt[x] == 1:
                    cnt[x] = 0
                else:
                    # 需要找一个 last[x] 之后的 ? 当成 I x
                    l = last[x]
                    before = bit_sum(bit, l - 1) if l > 0 else 0

                    if before == qcnt:
                        bad = i
                    else:
                        pos = bit_kth(bit, m, before + 1)
                        bit_add(bit, m, pos, -1)
                        qcnt -= 1
                        # cnt[x] 仍然是 0：
                        # ? 当 I x，当前 O x 又使用掉

                last[x] = i

        ans.append(str(bad))

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
## add your code here
import sys


def main():
    input = sys.stdin.readline

    n = int(input())
    points = []

    for _ in range(n):
        x, y = map(int, input().split())
        points.append((x, y))

    visited = [False] * n

    def dfs(u):
        visited[u] = True
        x1, y1 = points[u]

        for v in range(n):
            if not visited[v]:
                x2, y2 = points[v]
                if x1 == x2 or y1 == y2:
                    dfs(v)

    cnt = 0

    for i in range(n):
        if not visited[i]:
            cnt += 1
            dfs(i)

    print(cnt - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

#include <bits/stdc++.h>
using namespace std;

struct Chunk {
    int offset;
    string str;
    vector<int> pi;
};

struct Segment {
    string raw;
    int len;
    vector<Chunk> chunks;
    int anchor;
};

vector<int> buildPi(const string& p) {
    int n = p.size();
    vector<int> pi(n, 0);

    for (int i = 1; i < n; i++) {
        int j = pi[i - 1];

        while (j > 0 && p[i] != p[j]) {
            j = pi[j - 1];
        }

        if (p[i] == p[j]) {
            j++;
        }

        pi[i] = j;
    }

    return pi;
}

Segment buildSegment(const string& seg) {
    Segment res;
    res.raw = seg;
    res.len = seg.size();
    res.anchor = -1;

    int n = seg.size();
    int i = 0;

    while (i < n) {
        if (seg[i] == '?') {
            i++;
            continue;
        }

        int start = i;

        while (i < n && seg[i] != '?') {
            i++;
        }

        string part = seg.substr(start, i - start);
        Chunk c;
        c.offset = start;
        c.str = part;
        c.pi = buildPi(part);

        res.chunks.push_back(c);
    }

    for (int i = 0; i < (int)res.chunks.size(); i++) {
        if (res.anchor == -1 ||
            res.chunks[i].str.size() > res.chunks[res.anchor].str.size()) {
            res.anchor = i;
        }
    }

    return res;
}

bool checkAt(const Segment& seg, const string& s, int start) {
    int n = s.size();

    if (start < 0 || start + seg.len > n) {
        return false;
    }

    for (const auto& c : seg.chunks) {
        int pos = start + c.offset;

        if (pos < 0 || pos + (int)c.str.size() > n) {
            return false;
        }

        if (s.compare(pos, c.str.size(), c.str) != 0) {
            return false;
        }
    }

    return true;
}

int findSegment(const Segment& seg, const string& s, int minStart) {
    int n = s.size();

    if (minStart + seg.len > n) {
        return -1;
    }

    // 这一段全是 ?，直接放在最早位置即可
    if (seg.chunks.empty()) {
        return minStart;
    }

    const Chunk& anchor = seg.chunks[seg.anchor];

    int lower = minStart + anchor.offset;
    int upper = n - seg.len + anchor.offset;

    if (lower > upper) {
        return -1;
    }

    const string& p = anchor.str;
    const vector<int>& pi = anchor.pi;

    int j = 0;

    for (int i = lower; i < n; i++) {
        while (j > 0 && s[i] != p[j]) {
            j = pi[j - 1];
        }

        if (s[i] == p[j]) {
            j++;
        }

        if (j == (int)p.size()) {
            int anchorPos = i - (int)p.size() + 1;

            if (anchorPos > upper) {
                break;
            }

            int segStart = anchorPos - anchor.offset;

            if (segStart >= minStart && checkAt(seg, s, segStart)) {
                return segStart;
            }

            j = pi[j - 1];
        }
    }

    return -1;
}

bool matchWildcard(const string& pattern, const string& filename) {
    int n = filename.size();

    bool hasStar = false;
    for (char c : pattern) {
        if (c == '*') {
            hasStar = true;
            break;
        }
    }

    vector<Segment> segs;
    string cur;

    for (char c : pattern) {
        if (c == '*') {
            if (!cur.empty()) {
                segs.push_back(buildSegment(cur));
                cur.clear();
            }
        } else {
            cur.push_back(c);
        }
    }

    if (!cur.empty()) {
        segs.push_back(buildSegment(cur));
    }

    // 没有 *
    if (!hasStar) {
        if ((int)pattern.size() != n) {
            return false;
        }

        Segment only = buildSegment(pattern);
        return checkAt(only, filename, 0);
    }

    // 全是 *
    if (segs.empty()) {
        return true;
    }

    bool startStar = pattern[0] == '*';
    bool endStar = pattern.back() == '*';

    int pos = 0;
    int left = 0;
    int right = (int)segs.size() - 1;

    // 第一段必须匹配前缀
    if (!startStar) {
        if (!checkAt(segs[left], filename, 0)) {
            return false;
        }

        pos = segs[left].len;
        left++;
    }

    // 最后一段必须匹配后缀
    int suffixStart = -1;

    if (!endStar) {
        suffixStart = n - segs[right].len;

        if (suffixStart < pos) {
            return false;
        }

        if (!checkAt(segs[right], filename, suffixStart)) {
            return false;
        }

        right--;
    }

    // 中间段按顺序匹配
    for (int i = left; i <= right; i++) {
        int found = findSegment(segs[i], filename, pos);

        if (found == -1) {
            return false;
        }

        pos = found + segs[i].len;

        if (!endStar && pos > suffixStart) {
            return false;
        }
    }

    if (!endStar && pos > suffixStart) {
        return false;
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    int n;
    cin >> n;

    while (n--) {
        string filename;
        cin >> filename;

        cout << (matchWildcard(pattern, filename) ? "YES" : "NO") << '\n';
    }

    return 0;
}


## G 汉诺塔

## add your code here
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

ll qpow(ll a, int b) {
    ll res = 1;
    while (b--) {
        res *= a;
    }
    return res;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    map<string, int> rank;

    for (int i = 0; i < 6; i++) {
        string op;
        cin >> op;
        rank[op] = i;
    }

    char T, U;

    // 第一步只能是 AB 或 AC
    if (rank["AB"] < rank["AC"]) {
        T = 'B';
        U = 'C';
    } else {
        T = 'C';
        U = 'B';
    }

    string TU = "";
    TU += T;
    TU += U;

    string TA = "";
    TA += T;
    TA += 'A';

    string UA = "";
    UA += U;
    UA += 'A';

    string UT = "";
    UT += U;
    UT += T;

    ll ans;

    if (rank[TU] > rank[TA]) {
        // T -> U 优先级低于 T -> A
        ans = 2 * qpow(3, n - 1) - 1;
    } else {
        // T -> U 优先级高于 T -> A
        if (rank[UA] < rank[UT]) {
            ans = qpow(2, n) - 1;
        } else {
            ans = qpow(3, n - 1);
        }
    }

    cout << ans << '\n';

    return 0;
}

## H 马步距离

## add your code here
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    ll xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;

    ll dx = llabs(xp - xs);
    ll dy = llabs(yp - ys);

    ll x = max(dx, dy);
    ll y = min(dx, dy);

    if (x == 1 && y == 0) {
        cout << 3 << '\n';
        return 0;
    }

    if (x == 2 && y == 2) {
        cout << 4 << '\n';
        return 0;
    }

    ll ans = max((x + 1) / 2, (x + y + 2) / 3);

    if ((ans + x + y) % 2 != 0) {
        ans++;
    }

    cout << ans << '\n';

    return 0;
}

## I 直方图最大矩形

## add your code here
class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();
        stack<int> st;
        int ans = 0;

        // 遍历到 n，相当于最后补一个高度为 0 的柱子，用来清空栈
        for (int i = 0; i <= n; i++) {
            int cur = (i == n ? 0 : heights[i]);

            while (!st.empty() && cur < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();

                int left = st.empty() ? -1 : st.top();
                int width = i - left - 1;

                ans = max(ans, h * width);
            }

            st.push(i);
        }

        return ans;
    }
};

## J 消防局的设立

## add your code here
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<int> parent(n + 1, 0);
    vector<vector<int>> g(n + 1);

    for (int i = 2; i <= n; i++) {
        int p;
        cin >> p;

        parent[i] = p;
        g[i].push_back(p);
        g[p].push_back(i);
    }

    vector<int> depth(n + 1, 0);

    // 因为题目给的是每个点的父亲，正常情况下父亲编号小于儿子编号
    for (int i = 2; i <= n; i++) {
        depth[i] = depth[parent[i]] + 1;
    }

    vector<int> order;
    for (int i = 1; i <= n; i++) {
        order.push_back(i);
    }

    sort(order.begin(), order.end(), [&](int a, int b) {
        return depth[a] > depth[b];
    });

    vector<bool> covered(n + 1, false);

    auto cover = [&](int x) {
        covered[x] = true;

        for (int y : g[x]) {
            covered[y] = true;
        }

        for (int y : g[x]) {
            for (int z : g[y]) {
                covered[z] = true;
            }
        }
    };

    int ans = 0;

    for (int u : order) {
        if (covered[u]) {
            continue;
        }

        int x = u;

        // 往上走两步，能走多少走多少
        for (int k = 0; k < 2; k++) {
            if (parent[x] != 0) {
                x = parent[x];
            }
        }

        ans++;
        cover(x);
    }

    cout << ans << '\n';

    return 0;
}